In [ ]:
from partial_discharge_adaptive_fusion.protocol import assert_protocol_frozen, load_experiment_config
CONFIG = load_experiment_config('configs/experiments/two-dataset-confirmatory-v2-batch4-localraw.yaml')
assert_protocol_frozen(CONFIG)
assert CONFIG['experts']['batch_size'] == CONFIG['experts']['inference_batch_size'] == 4
print('Using config:', CONFIG['config_version'])

# Expert complementarity

**Objective.** Quantify temporal/CWT disagreements, exclusive successes and oracle headroom separately for MATLAB and VSB.

**Inputs.** Expert predictions generated under `two-dataset-confirmatory-v2-batch4-localraw` and labels from each dataset.

**Outputs.** Correctness categories, disagreement rates, oracle MCC and headroom tables.

**Experimental role.** Descriptive evidence before learned fusion.

**Leakage constraints.** Use only predictions generated under the v2 frozen protocol; do not use oracle information as a feature or deployable prediction.

In [ ]:
import numpy as np
from partial_discharge_adaptive_fusion.fusion import oracle_prediction
from partial_discharge_adaptive_fusion.evaluation import fast_mcc

def complementarity(labels, temporal_prediction, cwt_prediction, fixed_prediction=None):
    oracle = oracle_prediction(labels, temporal_prediction, cwt_prediction)
    row = {
        'both_correct': int(np.sum((temporal_prediction == labels) & (cwt_prediction == labels))),
        'both_wrong': int(np.sum((temporal_prediction != labels) & (cwt_prediction != labels))),
        'temporal_only': int(np.sum((temporal_prediction == labels) & (cwt_prediction != labels))),
        'cwt_only': int(np.sum((temporal_prediction != labels) & (cwt_prediction == labels))),
        'oracle_mcc': fast_mcc(labels, oracle),
    }
    if fixed_prediction is not None:
        row['oracle_headroom_vs_fixed'] = fast_mcc(labels, oracle) - fast_mcc(labels, fixed_prediction)
    return row

print('Run complementarity separately for each frozen dataset/test partition.')

## Findings and handoff

Complementarity supports the existence of recoverable expert disagreement, not the superiority of a learned fusion.

**Next stage:** evaluate fixed and adaptive fusion with development-only selection.